# Honey Bee Video Motion-Regime Experiments

This notebook documents exploratory motion-regime annotation experiments for the Smith honey bee hive video. The goal is not to identify individual bees perfectly. The goal is to make local motion regimes visible for human review, especially regimes that appear, disappear, or spatially organize around comb/festoon areas.

The current working input for these experiments is the resequenced artifact:

`data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4`

The original source video is retained under `data/raw/`, but the experiments below use the resequenced version unless otherwise noted.


## Shared Inputs and Conventions

Common tools:

- Preset runner: `src/analyze/run_analysis.py`
- Direct annotator: `src/analyze/annotate_motion_regimes.py`
- Sample runner: `src/analyze/run_motion_regime_samples.py`
- Overnight sampler tour: `src/pipeline/exp3_overnight.py`
- Quadrant score script: `src/pipeline/score_exp3_quadrants.py`
- Output root: `data/qc/`

The annotation scripts create:

- `motion_regime_features.csv`: one row per grid cell per time window, including regime probabilities.
- `motion_regime_overlay.mp4`: visual overlay for human review.
- `metadata.json`: exact run settings.

Interpretation:

- Color is an unsupervised local motion-regime cluster.
- Arrows show mean local optical-flow vectors.
- Labels are exploratory annotations over local image regions, not behavior truth labels and not bee identities.


In [ ]:
from pathlib import Path

VIDEO = Path("data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4")
QC = Path("data/qc")

print(VIDEO.resolve())
print(QC.resolve())


## Experiment 1: Baseline Local Motion Regimes

<img src="../data/qc/exp1_reseq_2min_v0p1/exp1.png" width="400" />

### Question

Can local optical-flow features over a five-second history window produce visually meaningful regime annotations over the hive surface?

### Inputs

- Preset: `exp1_reseq_2min_v0p1`
- Input video: `data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4`
- Frame range: `start_frame=0`, `duration_frames=3000`
- Window length: `125` frames
- Stride: `25` frames
- Grid: `16x16`
- Clusters: `6`
- Feature set: `exp1`, the pre-angular-neighbor baseline feature set
- PCA: disabled
- Optical-flow scale width: `412`

### Result

Experiment 1 established the basic overlay machinery and produced some visually interpretable regional structure, but it did not provide a strong enough separation criterion for choosing motion-annotation parameters.


In [ ]:
%%bash
uv run python src/analyze/run_analysis.py   exp1_reseq_2min_v0p1   --video data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4   --out data/qc/exp1_reseq_2min_v0p1


## Experiment 2: Angular/Neighbor-Synchrony Features

### Question

Can local angular features separate visually distinct coordinated motion regimes, especially comb-area behavior where neighboring bees appear to oscillate within a shared restricted range of directions?

### Inputs

- Preset: `exp2_reseq_focus_v0p1`
- Frame range: `start_frame=14500`, `duration_frames=3000`
- Window length: `125` frames
- Stride: `25` frames
- Grid: `32x32`
- Clusters: `8`
- Feature set: `full`
- PCA components: `8`
- Optical-flow scale width: `412`
- Angular and neighbor feature weights: `1.0`

### Methods

Experiment 2 added angular and neighbor-synchrony measurements to the clustering matrix, including `direction_concentration`, `angular_sweep_std`, `angular_sweep_abs_mean`, `neighbor_angular_sweep_abs_diff`, and `neighbor_direction_concentration_diff`.

### Result

Experiment 2 was scientifically useful but visually tepid. It did not recover the strong regional segregation seen in earlier exploratory runs, and it did not yet give a reliable parameter-selection metric.


In [ ]:
%%bash
uv run python src/analyze/run_analysis.py   exp2_reseq_focus_v0p1   --video data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4   --out data/qc/exp2_reseq_focus_v0p1


## Experiment 3: Overnight Parameter Sweep and Frame-87950 Metric

### Question

Can sampled overlays plus a simple quantitative score identify motion-annotation settings that better separate visually distinct hive regions?

### Background

After Experiments 1 and 2, the main problem was no longer only how to generate overlays. The main problem was how to evaluate parameter choices without relying entirely on side-by-side visual judgment. The overnight sweep tested a set of deliberately distinct sampled settings across the resequenced video.

The best side-by-side visual candidates were `16_long_window` and `11_high_angular`; `16_long_window` was the strongest reviewed result from the overnight tour.

### Sampling Design

- Script: `src/pipeline/exp3_overnight.py`
- Base preset: `exp3_sampler`
- Input video: `data/artifacts/resequenced/reseq_1_start04__20190609_175013_side0_top.mp4`
- Samples per run: `10`
- Default sample length: `250` frames, except the long-window condition uses `500` frames
- Default window length: `125` frames, except short/long window ablations
- Default stride: `1`
- Default grid: `32x32`
- Default clusters: `8`
- Default feature set: `full`
- Default velocity transform: `raw`
- Default angular/neighbor weights: `2.0` and `1.5`
- Output root: `data/qc/exp3_overnight/`

### Evaluation Metric

We chose frame `87950` as a concrete review frame. For each run, the scoring script finds the sample containing that frame, selects the nearest available feature window, and compares clustering in the upper-right quadrant against the lower-right quadrant.

Reported measures:

- `Avg label diff`: absolute difference between the mean cluster label in the upper-right and lower-right quadrants. This is the simple success metric used for quick ranking.
- `Prob. TV`: total-variation distance between the average cluster-probability profiles of the two quadrants.
- `Cluster TV`: total-variation distance between hard cluster-label distributions in the two quadrants.

Cluster labels are categorical, so `Prob. TV` and `Cluster TV` are more defensible than mean-label difference. In practice, the mean-label score matched the visual review well enough to guide the next sweep.

### Results Table

| Rank | Run | Grid | Clusters | PCA | Feature set | Velocity | Activity | Angular | Neighbor | Window | Avg label diff | Prob. TV | Cluster TV |
|---:|---|---:|---:|---:|---|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | `16_long_window` | 32x32 | 8 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 250 | 1.320 | 0.428 | 0.430 |
| 2 | `11_high_angular` | 32x32 | 8 | 0 | full | raw | 0.30 | 3.0 | 1.5 | 125 | 0.930 | 0.425 | 0.422 |
| 3 | `05_clusters10` | 32x32 | 10 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 125 | 0.922 | 0.449 | 0.453 |
| 4 | `04_clusters6` | 32x32 | 6 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 125 | 0.840 | 0.431 | 0.426 |
| 5 | `02_grid48_pca0` | 48x48 | 8 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 125 | 0.816 | 0.414 | 0.417 |
| 6 | `07_activity050` | 32x32 | 8 | 0 | full | raw | 0.50 | 2.0 | 1.5 | 125 | 0.773 | 0.394 | 0.406 |
| 7 | `09_asinh_velocity` | 32x32 | 8 | 0 | full | asinh | 0.30 | 2.0 | 1.5 | 125 | 0.707 | 0.421 | 0.422 |
| 8 | `06_activity020` | 32x32 | 8 | 0 | full | raw | 0.20 | 2.0 | 1.5 | 125 | 0.621 | 0.433 | 0.434 |
| 9 | `13_exp1_features` | 32x32 | 8 | 0 | exp1 | raw | 0.30 | 2.0 | 1.5 | 125 | 0.613 | 0.400 | 0.406 |
| 10 | `12_high_neighbor` | 32x32 | 8 | 0 | full | raw | 0.30 | 2.0 | 3.0 | 125 | 0.570 | 0.407 | 0.402 |
| 11 | `08_log1p_velocity` | 32x32 | 8 | 0 | full | log1p | 0.30 | 2.0 | 1.5 | 125 | 0.449 | 0.442 | 0.449 |
| 12 | `14_beginner_features` | 32x32 | 6 | 0 | beginner | raw | 0.30 | 2.0 | 1.5 | 125 | 0.406 | 0.374 | 0.371 |
| 13 | `15_short_window` | 32x32 | 8 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 50 | 0.164 | 0.445 | 0.434 |
| 14 | `10_low_group_weights` | 32x32 | 8 | 0 | full | raw | 0.30 | 1.0 | 1.0 | 125 | 0.059 | 0.426 | 0.418 |
| 15 | `03_grid64_pca0` | 64x64 | 8 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 125 | 0.029 | 0.427 | 0.427 |
| 16 | `01_grid32_pca0` | 32x32 | 8 | 0 | full | raw | 0.30 | 2.0 | 1.5 | 125 | 0.023 | 0.435 | 0.445 |

### Result

Experiment 3 successfully produced both a visually promising condition and a practical evaluation metric. The long-window setting (`16_long_window`) was the strongest reviewed result and also ranked first by the frame-87950 upper-right/lower-right mean-label difference. The high-angular setting (`11_high_angular`) was also strong.

The next step is a larger parameter sweep focused around this frame and metric, with the metric adapted to additional review frames or biological conditions as needed.


In [ ]:
%%bash
# Run the overnight tour.
uv run python src/pipeline/exp3_overnight.py

# Score upper-right/lower-right separation at the selected review frame.
uv run python src/pipeline/score_exp3_quadrants.py   --root data/qc/exp3_overnight   --target-frame 87950


## Next Step

Use the Experiment 3 metric to run a larger focused sweep. The immediate search should vary parameters around the best overnight conditions rather than treating the whole parameter space as equally promising.

Priority directions:

- Expand around `16_long_window`: longer history windows, longer samples, and nearby window/sample ratios.
- Expand around `11_high_angular`: angular-feature weights near `3.0`, with neighbor weights held moderate.
- Keep `grid32` and `grid48` in play; `grid64` performed poorly on the selected metric.
- Score the same run outputs at additional review frames before treating frame `87950` as sufficient.
- Prefer distributional scores (`Prob. TV`, `Cluster TV`) when the goal is a publication-grade metric; keep mean-label difference as a quick visual-search heuristic.
